In [3]:
from bs4 import BeautifulSoup as soup
from bs4 import Tag, NavigableString
import mwparserfromhell
import requests
import pandas
import re

In [4]:
# Test code for downloaded file wiki-'eye'.html
# request page for eye online

with open("wiki-'eye'.html", 'r', encoding='utf-8') as f:
    page = f.read()
soupify = soup(page, 'html.parser')
# wikicode = mwparserfromhell.parse(soupify)
# mw-heading 2 for language, 
#   mw-heading 3 for subheadings like etymology, pronunciation, etc. (look at next element: paragraph, list, etc.)
    # span class="IPA nowrap" for pronunciation?
        # pronunciation can be mw-heading 4???
            # mayhaps id contains "Pronunciation"?, then look for list element
            # thus, id contains "Etymology" for etymology section?
    # span class="etyl" for etymology links (Ole English, Middle English)
    # i class="Latn" for Latin script (for non-English words), mention class in the etymology section
    # WHY ISN'T THE WIKIMEDIA PAGE NESTED AT ALL!!!?!?!!    
# wikicode.filter_templates()
soupify.find_all('h3') # seemingly gets relevant headings for this page

# feels a bit specific but worth a shot
soupify.find_all('h3')[0].parent.next_sibling.next_sibling.next_element.find('span', class_='IPA').text # get IPA pronuncation from first h3 heading (pronunciation) and next sibling (list of pronunciations)'

# okay, let's try parsing the etymology section, which is the second h3 heading
all_etyl = soupify.find_all('h3')[1].parent.next_sibling.next_sibling.find_all('i', class_='Latn')
[x.text for x in all_etyl]

# unfortunately, we cannot get the related term "ogle" at all - no ids or anything

['eye', 'yë', 'eyghe', 'ēage', '*augā', '*augô', '*h₃okʷ-', '*h₃ekʷ-']

In [18]:
def get_etym_links(etym_p): 
    # etym_links structure: {language: [{"word": word, "link": link}, ...], ...}
    etym_links = {"Unknown Language": []}#, "Other Tag" : [],}
    language = None
    for element in etym_p.descendants:
        # print("Curr: ", language)
        # print("Element: ", element)
        if isinstance(element, NavigableString) or not element.get("class"): 
            continue
        # if element is a "language" link, establish language key in dict 
        elif element.name == "span" and "etyl" in element.get("class"):
            a_tag = element.find("a")
            # language = element.next.text.replace(" ", "_")
            language = a_tag.text.replace(" ", "_") if a_tag else None
            if language and language not in etym_links:
                etym_links[language] = []
            # skip to next unnested element
            

        # if element is a latin-script word, add to dict with language key if possible, otherwise add to "Unknown" key; if it's a link, add the link as well
        elif element.name == "i" and "Latn" in element.get("class"):
            # link = None
            # temp = element.next_sibling
            # print(temp)
            # while temp and not link:
            #     temp = temp.next_sibling
            #     if isinstance(temp, Tag):
            #         link = temp
            # print("link, temp: ", link, temp)
            # print("Element: ", element, "Link: ", link)
            link = element.find_next("a")
            word_dict = {"word": element.text, "link" : "https://en.wiktionary.org/{}".format(link.get("href")) if link and link.name == "a" else None}
            
            if language:
                etym_links[language].append(word_dict)
            
            else:
                etym_links["Unknown Language"].append(word_dict)
        # if element is a link but we don't have a language established, add to "Other" key with link if possible
        # come back to this, messes up with language links
        elif element.name == "a" and language:
            continue
            # word_dict = {"word": element.text, "link" : element.get("href")}
            # etym_links["Other Tag"].append(word_dict)
    if etym_links["Unknown Language"] == []:
        etym_links.pop("Unknown Language")
    # if etym_links.get("Other Tag") and etym_links["Other Tag"] == []:
    #     etym_links.pop("Other Tag")
    return etym_links

In [6]:
def get_pronunciation(pron_ul): 
    # for li in pron_ul.find_all("li"):
        # if li.find("span", class_="IPA nowrap"):
        #     return li.find("span", class_="IPA nowrap").text
        return pron_ul.find("span", class_="IPA nowrap").text if pron_ul.find("span", class_="IPA nowrap") else None

In [7]:
def parse_etym_links(etym_links): 
    return

In [8]:
def filter_language_sections(page_soup, language): 
    headers = page_soup.find_all("h2")
    # 
    language_section_idx = headers.index(page_soup.find("h2", id=language))
    language_section = headers[language_section_idx]

    relevant_html = []    
    for html in language_section.find_all_next():
        if html.name == "h2" and html != language_section:
            break
        if isinstance(html, Tag):
            relevant_html.append(html)    
            
    return soup("".join(str(html) for html in relevant_html), "html.parser")

In [27]:
# Scrapes Wiktionary page for relevant links, etymology, pronunciation, compacts into pandas df
def scrape_wiktionary_page(html_file):
    
    # url = f"https://en.wiktionary.org/wiki/{word}"
    # page = requests.get(url)
    # print(html_file)
    if html_file.startswith("http"):
        grabbed_html = grab_page_html(html_file)
        html_content = grabbed_html#.decode("utf-8")
        word = html_file.split("/")[-1].split(".")[0] # get word from file name
    else:
        with open(html_file, "r", encoding="utf-8") as f:
            html_content = f.read()
        word = html_file.split(".")[0].split("-")[-1] # get word from file name
    page_soup = soup(html_content, "html.parser")

    relevant_html = filter_language_sections(page_soup, "English")
    # print("Relevant html: ", relevant_html)

    # grab all etymology sections that have id containing "Etymology" that are in the english-language section (some words have multiple etymologies)
    # we should really regex the etymology paragraph to try and grab non-hyperlink info like "From", "Related to", "Cognate with", etc. and the language of origin if possible
    # Currently just grabs the hyperlinks and their associated languages if possible
    etymology = relevant_html.find_all("h3", id=re.compile("Etymology"))
    etym_dict = {}
    if etymology:
        for etym in etymology:
            if etym.find_next("p"):
                etym_dict[etym.text] = get_etym_links(etym.find_next("p"))

    # Get pronunciation
    pronunciation = relevant_html.find("h3", id="Pronunciation")

    if pronunciation:
        pronunciation_ul = pronunciation.find_next("ul")
        pronunciation_text = get_pronunciation(pronunciation_ul)
    else:
        pronunciation_text = None

    # # Get related links (e.g., synonyms, antonyms, related terms)
    # related_links = []
    # for section in page_soup.find_all("h3", class_="mw-headline"):
    #     if section.text in ["Synonyms", "Antonyms", "Related terms" ]:
    #         links = section.find_next("ul").find_all("a")
    #         related_links.extend([link.text for link in links])

    # Compile into pandas DataFrame
    data = {
        "English Word": word,
        "Etymologies": etym_dict,
        "Pronunciation": pronunciation_text,
        # "Related Links": related_links
        # "Further Enquiry" : 
    }
    
    df = pandas.DataFrame([data])
    return df
    
def grab_page_html(word):
    url = f"https://en.wiktionary.org/wiki/{word}"
    page = requests.get(url)
    return page.content

In [45]:
df = scrape_wiktionary_page("wiki-'eye'.html")
df

,English Word,Etymologies,Pronunciation
0,'eye',{'Etymology 1': {'Middle_English': [{'word': '...,/aɪ/


In [22]:
# 200 ihops is one really good tattoo btw 
# df["Etymology"][0]
# for item in df["Etymology"][0]:
    # print(item)
    # for x in item:
    #     print(x, item[x])
for key, val in df["Etymologies"][0]["Etymology 1"].items():
    for item in val:
        print(key, item)

# df["Etymology"][0]

Middle_English {'word': 'eye', 'link': 'https://en.wiktionary.org/#Middle_English:_eye'}
Middle_English {'word': 'yë', 'link': 'https://en.wiktionary.org//wiki/ye#Middle_English'}
Middle_English {'word': 'eyghe', 'link': 'https://en.wiktionary.org//wiki/eyghe#Middle_English'}
Old_English {'word': 'ēage', 'link': 'https://en.wiktionary.org//wiki/eage#Old_English'}
Proto-West_Germanic {'word': '*augā', 'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-West_Germanic/aug%C4%81'}
Proto-Germanic {'word': '*augô', 'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-Germanic/aug%C3%B4'}
Proto-Indo-European {'word': '*h₃okʷ-', 'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-Indo-European/h%E2%82%83ok%CA%B7-'}
Proto-Indo-European {'word': '*h₃ekʷ-', 'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-Indo-European/h%E2%82%83ek%CA%B7-'}


In [26]:
df.iloc[0]["Etymologies"]

{'Etymology 1': {'Middle_English': [{'word': 'eye',
    'link': 'https://en.wiktionary.org/#Middle_English:_eye'},
   {'word': 'yë', 'link': 'https://en.wiktionary.org//wiki/ye#Middle_English'},
   {'word': 'eyghe',
    'link': 'https://en.wiktionary.org//wiki/eyghe#Middle_English'}],
  'Old_English': [{'word': 'ēage',
    'link': 'https://en.wiktionary.org//wiki/eage#Old_English'}],
  'Proto-West_Germanic': [{'word': '*augā',
    'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-West_Germanic/aug%C4%81'}],
  'Proto-Germanic': [{'word': '*augô',
    'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-Germanic/aug%C3%B4'}],
  'Proto-Indo-European': [{'word': '*h₃okʷ-',
    'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-Indo-European/h%E2%82%83ok%CA%B7-'},
   {'word': '*h₃ekʷ-',
    'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-Indo-European/h%E2%82%83ek%CA%B7-'}]},
 'Etymology 3': {'Unknown Language': [{'word': 'nye',
    'link': 'htt

In [58]:
copy = df.copy()

In [61]:
# get rid of useless etymologies
for entry in df.index:
    etymologies = df.iloc[entry]["Etymologies"]
    print("Etym:", etymologies)
    for etym in etymologies:
        # no etymologies present
        if len(etymologies[etym]) == 0:
            etymologies.pop(etym)
        # only etymologies with unknown language present
        elif list(etymologies[etym].keys()) == ["Unknown Language"]:
            etymologies.pop(etym)

Etym: {'Etymology 1': {'Middle_English': [{'word': 'eye', 'link': 'https://en.wiktionary.org/#Middle_English:_eye'}, {'word': 'yë', 'link': 'https://en.wiktionary.org//wiki/ye#Middle_English'}, {'word': 'eyghe', 'link': 'https://en.wiktionary.org//wiki/eyghe#Middle_English'}], 'Old_English': [{'word': 'ēage', 'link': 'https://en.wiktionary.org//wiki/eage#Old_English'}], 'Proto-West_Germanic': [{'word': '*augā', 'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-West_Germanic/aug%C4%81'}], 'Proto-Germanic': [{'word': '*augô', 'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-Germanic/aug%C3%B4'}], 'Proto-Indo-European': [{'word': '*h₃okʷ-', 'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-Indo-European/h%E2%82%83ok%CA%B7-'}, {'word': '*h₃ekʷ-', 'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-Indo-European/h%E2%82%83ek%CA%B7-'}]}}


In [62]:
df.iloc[0]["Etymologies"]

{'Etymology 1': {'Middle_English': [{'word': 'eye',
    'link': 'https://en.wiktionary.org/#Middle_English:_eye'},
   {'word': 'yë', 'link': 'https://en.wiktionary.org//wiki/ye#Middle_English'},
   {'word': 'eyghe',
    'link': 'https://en.wiktionary.org//wiki/eyghe#Middle_English'}],
  'Old_English': [{'word': 'ēage',
    'link': 'https://en.wiktionary.org//wiki/eage#Old_English'}],
  'Proto-West_Germanic': [{'word': '*augā',
    'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-West_Germanic/aug%C4%81'}],
  'Proto-Germanic': [{'word': '*augô',
    'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-Germanic/aug%C3%B4'}],
  'Proto-Indo-European': [{'word': '*h₃okʷ-',
    'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-Indo-European/h%E2%82%83ok%CA%B7-'},
   {'word': '*h₃ekʷ-',
    'link': 'https://en.wiktionary.org//wiki/Reconstruction:Proto-Indo-European/h%E2%82%83ek%CA%B7-'}]}}

In [63]:
df.to_json("wiktionary_data.json", orient="records")